# Projeto 2 — Extrator Clínico Neuro-Simbólico
## Processamento de Línguas Naturais (PLN 2026 / UNICAMP)

> **Professor:** André Santanchè  
> **Equipe:** Caio (248243), Fábio Luiz (247079), Gustavo Sousa (247149), T.H. de Camargo J. (206235), Wallysson (200061)

---
### Visão Geral da Demonstração Executável (Narrativa Científica em 3 Atos)
Este notebook demonstra a evolução científica completa do Projeto 2, superando as regras do Projeto 1 **sem recorrer a LLMs, Transformers ou embeddings pré-treinados**:
1. **Fase 1:** Mineração Estatística de N-Grams (PPMI e Student's $t$-test), Embeddings Densos SVD 64d (Levy & Goldberg / Firth 1957) e Classificação Morfológica de Entidades OOV.
2. **Fase 2:** Supervisão Fraca com Trigger-Masking e N-Grams Posicionais (`PRE_`, `BTW_`, `POST_`) particionados por paciente (StratifiedGroupKFold).
3. **Ato 1 (Diagnóstico Forense):** Torneio de Modelos Clássicos e descoberta das 3 anomalias matemáticas em torno do muro do `0.5040`.
4. **Ato 2 (As 5 Viradas de Jogo BioNLP):** N-grams de Clusters Semânticos, Cascata Hierárquica, Confident Learning (47 ruídos do P1 expurgados), DistMult-MLP Trilinear de 3 Torres e Fechamento de Triângulos Clínicos no Grafo.
5. **Ato 3 (Fusão Precoce & Ensemble Neuro-Simbólico):** Resolução do ponto cego de Teoria da Informação via assinaturas ontológicas (`SIG_E1_TO_E2`), buckets de distância sintática e Ensemble Calibrado com subida de encosta (Macro-F1 salta para `0.6173`, Recall `0.6929`).
6. **Visualizador & Sandbox Contrafactual Ao Vivo:** Demonstração interativa What-If no navegador.

In [ ]:
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path

# Ajuste de paths relativos
current_dir = Path.cwd()
project2_dir = current_dir.parent.parent
print("Diretório do Projeto 2:", project2_dir)


## 1. Fase 1: Mineração de Collocations Clínicas e Embeddings PPMI-SVD 64d
Extração de bigramas e trigramas com alta *Positive Pointwise Mutual Information* (PPMI) e significância estatística ($t$-score).

In [ ]:
colloc_path = project2_dir / 'data' / 'interim' / 'collocations_pmi.csv'
colloc_df = pd.read_csv(colloc_path)
print(f"Total de collocations mineradas no MultiCaRe: {len(colloc_df)}")
colloc_df.head(10)[['ngram_text', 'n_size', 'freq', 'ppmi_score', 't_score', 'composite_score']]

## 2. Tipagem Morfológica de Entidades Out-of-Vocabulary (OOV)
Classificador linear treinado sobre TF-IDF de subpalavras (3 a 5 caracteres) reconhece radicais greco-latinos (`-itis`, `-ectomy`, `-emia`, `-oma`, `-scopy`) e promove novos nós médicos para o grafo.

In [ ]:
oov_path = project2_dir / 'data' / 'interim' / 'oov_promoted_nodes.csv'
oov_df = pd.read_csv(oov_path)
print(f"Total de nós OOV promovidos estatisticamente: {len(oov_df)}")
oov_df.head(10)[['node_id', 'label', 'type', 'discovery_method']]

## 3. Fase 2: Dataset Silver Standard com Trigger-Masking e N-Grams de Clusters

In [ ]:
## 4. O Torneio de Modelos em 3 Atos Científicos (Ablação 5-Fold por Paciente)
- **Ato 1:** Mostra os baselines clássicos e o diagnóstico das anomalias (`0.5040`).
- **Ato 2:** Demonstra os desbloqueios das 5 inovações (Three-Tower DistMult-MLP e Cascata Denoised).
- **Ato 3:** Demonstra o rompimento definitivo do teto de informação através da tipagem ontológica precoce (`0.5385`) e Ensemble Neuro-Simbólico (`0.6173`).

## 4. O Torneio de Modelos em 2 Atos Científicos (Ablação 5-Fold por Paciente)
- **Ato 1:** Mostra os baselines clássicos e o diagnóstico das anomalias (`0.5040`).
- **Ato 2:** Demonstra os desbloqueios das 5 inovações (Three-Tower DistMult-MLP e Cascata Denoised).

In [ ]:
benchmark_path = project2_dir / 'data' / 'processed' / 'ablation_benchmark.json'
with open(benchmark_path, 'r', encoding='utf-8') as f:
    benchmark = json.load(f)

rows = []
for b in benchmark:
    rows.append({
        'Ato': b.get('act', 'Ato 1 (Baseline)'),
        'Experimento / Modelo': b.get('experiment_id', b.get('model_name')),
        'N-Gram Range': str(b.get('ngram_range')),
        'Ponderação / Features': b.get('weighting'),
        'Macro-Precision': b.get('macro_precision'),
        'Macro-Recall': b.get('macro_recall'),
        'Macro-F1': b.get('macro_f1')
    })

df_bench = pd.DataFrame(rows)
df_bench

## 5. Auditoria de Ruído com Confident Learning (Expurgo de Erros do P1)
O modelo estatístico do P2 auditou e limpou **47 falsos positivos de coocorrência** gerados pelas regras do P1.

In [ ]:
nodes_p1 = len(pd.read_csv(project2_dir.parent / 'project1' / 'data' / 'output' / 'nodes.csv'))
edges_p1 = len(pd.read_csv(project2_dir.parent / 'project1' / 'data' / 'output' / 'edges.csv'))
nodes_p2 = len(pd.read_csv(project2_dir / 'data' / 'processed' / 'nodes_p2.csv'))
edges_p2 = len(pd.read_csv(project2_dir / 'data' / 'processed' / 'edges_p2.csv'))

summary_kg = pd.DataFrame([
    {'Métrica': 'Total de Nós', 'Projeto 1 (Simbólico)': nodes_p1, 'Projeto 2 (Neuro-Simbólico)': nodes_p2, 'Variação': f'+{nodes_p2 - nodes_p1} (+{((nodes_p2 - nodes_p1)/nodes_p1)*100:.2f}%)'},
    {'Métrica': 'Total de Arestas', 'Projeto 1 (Simbólico)': edges_p1, 'Projeto 2 (Neuro-Simbólico)': edges_p2, 'Variação': f'+{edges_p2 - edges_p1} (+{((edges_p2 - edges_p1)/edges_p1)*100:.2f}%)'},
    {'Métrica': 'Arestas Descobertas por ML', 'Projeto 1 (Simbólico)': 0, 'Projeto 2 (Neuro-Simbólico)': edges_p2 - edges_p1, 'Variação': 'Inéditas (P_ML >= 0.70)'},
    {'Métrica': 'Triângulos Clínicos Fechados', 'Projeto 1 (Simbólico)': 0, 'Projeto 2 (Neuro-Simbólico)': 445, 'Variação': 'Relational Belief Propagation'},
    {'Métrica': 'Arestas com Confiança Calibrada', 'Projeto 1 (Simbólico)': '0% (Binárias)', 'Projeto 2 (Neuro-Simbólico)': '100% (P in [0, 1])', 'Variação': 'Parametrizável via Slider'}
])
summary_kg


## 7. Visualizador Interativo SOTA: Linha do Tempo DAG, Narrativa & Sandbox What-If
Abra o arquivo HTML gerado em `project2/data/processed/graph_p2_visualization.html` (ou `project2/assets/graph_p2_visualization.html`) no seu navegador:
- **100% Offline:** Cytoscape.js embutido localmente, sem dependência de internet ou CDN.
- **Linha do Tempo DAG (Inspirada no P1):** Nós posicionados horizontalmente no eixo X pela sequência cronológica de suas sentenças ($t = 0 \dots N$), permitindo ler o caso da admissão ao desfecho.
- **Player Cronológico (Evolução Clínica):** Controles `Play / Pause / Passo a Passo` e Slider para assistir ao crescimento do grafo conforme o relato médico se desenrola.
- **Rastreabilidade de Frases:** Narrativa clínica anotada com `<mark>` e badges para as novas relações descobertas por ML, com sincronização bidirecional texto <-> grafo.
- **Slider Dinâmico ($\tau$):** Filtre arestas por confiança em tempo real.
- **Sandbox Contrafactual What-If:** Clique em qualquer aresta, altere a frase na caixa de texto (ex: `notable for` $\to$ `negative for`) e veja a aresta mudar de cor e classe instantaneamente no grafo!


In [ ]:
nodes_p1 = len(pd.read_csv(project2_dir.parent / 'project1' / 'data' / 'output' / 'nodes.csv'))
edges_p1 = len(pd.read_csv(project2_dir.parent / 'project1' / 'data' / 'output' / 'edges.csv'))
nodes_p2 = len(pd.read_csv(project2_dir / 'data' / 'processed' / 'nodes_p2.csv'))
edges_p2 = len(pd.read_csv(project2_dir / 'data' / 'processed' / 'edges_p2.csv'))

summary_kg = pd.DataFrame([
    {'Métrica': 'Total de Nós', 'Projeto 1 (Simbólico)': nodes_p1, 'Projeto 2 (Neuro-Simbólico)': nodes_p2, 'Variação': f'+{nodes_p2 - nodes_p1} (+{((nodes_p2 - nodes_p1)/nodes_p1)*100:.2f}%)'},
    {'Métrica': 'Total de Arestas', 'Projeto 1 (Simbólico)': edges_p1, 'Projeto 2 (Neuro-Simbólico)': edges_p2, 'Variação': f'+{edges_p2 - edges_p1} (+{((edges_p2 - edges_p1)/edges_p1)*100:.2f}%)'},
    {'Métrica': 'Arestas Descobertas por ML', 'Projeto 1 (Simbólico)': 0, 'Projeto 2 (Neuro-Simbólico)': edges_p2 - edges_p1, 'Variação': 'Inéditas (P_ML >= 0.70)'},
    {'Métrica': 'Triângulos Clínicos Fechados', 'Projeto 1 (Simbólico)': 0, 'Projeto 2 (Neuro-Simbólico)': 438, 'Variação': 'Relational Belief Propagation'},
    {'Métrica': 'Arestas com Confiança Calibrada', 'Projeto 1 (Simbólico)': '0% (Binárias)', 'Projeto 2 (Neuro-Simbólico)': '100% (P in [0, 1])', 'Variação': 'Parametrizável via Slider'}
])
summary_kg

## 7. Visualizador Interativo Cytoscape.js com Sandbox Contrafactual Ao Vivo
Abra o arquivo HTML gerado em `project2/data/processed/graph_p2_visualization.html` (ou `project2/assets/graph_p2_visualization.html`) no seu navegador:
- **100% Offline:** Sem dependência de CDN ou servidores web externos.
- **Slider Dinâmico ($\tau$):** Filtre arestas por confiança em tempo real.
- **Simulador What-If:** Clique em qualquer aresta, altere a frase na caixa de texto (ex: `notable for` $\to$ `negative for`) e veja a aresta mudar de cor e classe instantaneamente no grafo!